In [ ]:
"""" Importing all libraries """
import librosa
import librosa.display
import matplotlib.pyplot as plt
import kagglehub

import os
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras import layers, models

from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

In [ ]:
# Download UrbanSound8K from Kaggle
dataset_path = kagglehub.dataset_download("chrisfilo/urbansound8k")

print("Dataset downloaded to:", dataset_path)

Using Colab cache for faster access to the 'urbansound8k' dataset.
Dataset downloaded to: /kaggle/input/urbansound8k


In [ ]:
""" Understanding the dataset """

metadata_path = os.path.join(dataset_path, "UrbanSound8K.csv")

metadata = pd.read_csv(metadata_path)

metadata.head()


,slice_file_name,fsID,start,end,salience,fold,classID,class
0,100032-3-0-0.wav,100032,0.0,0.317551,1,5,3,dog_bark
1,100263-2-0-117.wav,100263,58.5,62.500000,1,5,2,children_playing
2,100263-2-0-121.wav,100263,60.5,64.500000,1,5,2,children_playing
3,100263-2-0-126.wav,100263,63.0,67.000000,1,5,2,children_playing
4,100263-2-0-137.wav,100263,68.5,72.500000,1,5,2,children_playing


In [ ]:
# <TODO> Print the total audio records present in the dataset

# <TODO> Print the number of audio files available for each class

In [ ]:
# Helper function to load the paths of audio files
def get_audio_path(row):
    fold = "fold" + str(row["fold"])
    filename = row["slice_file_name"]

    return os.path.join(dataset_path, fold, filename)

metadata["audio_path"] = metadata.apply(get_audio_path, axis=1)

metadata.head()

,slice_file_name,fsID,start,end,salience,fold,classID,class,audio_path
0,100032-3-0-0.wav,100032,0.0,0.317551,1,5,3,dog_bark,/kaggle/input/urbansound8k/fold5/100032-3-0-0.wav
1,100263-2-0-117.wav,100263,58.5,62.500000,1,5,2,children_playing,/kaggle/input/urbansound8k/fold5/100263-2-0-11...
2,100263-2-0-121.wav,100263,60.5,64.500000,1,5,2,children_playing,/kaggle/input/urbansound8k/fold5/100263-2-0-12...
3,100263-2-0-126.wav,100263,63.0,67.000000,1,5,2,children_playing,/kaggle/input/urbansound8k/fold5/100263-2-0-12...
4,100263-2-0-137.wav,100263,68.5,72.500000,1,5,2,children_playing,/kaggle/input/urbansound8k/fold5/100263-2-0-13...


In [ ]:
# <TODO>: Complete the helper function to load audio files
def load_audio_file(file_path, target_sample_rate=22050, duration=4):
    audio, sr = None # Load the audio file using librosa

    # Target length = 4 seconds
    target_length = target_sample_rate * duration

    # Pad if audio is shorter than 4 seconds
    if len(audio) < target_length:
        padding = target_length - len(audio)
        audio = np.pad(audio, (0, padding))

    # Trim if audio is longer than 4 seconds
    else:
        audio = audio[:target_length]

    return audio, sr

In [ ]:
#<TODO>: Try loading the first audio file


#<TODO>: Visualize the loaded audio using librosa



In [ ]:
from IPython.display import Audio

#<TODO> Find a way to display and play the first audio file


In [ ]:
#<TODO>: Complete the function to help trim the audio
def trim_audio(audio, top_db=20):

    # Trim audio
    trimmed_audio, index = None #Todo

    return trimmed_audio, index

#<TODO>: Load and Trim the first audio

#<TODO>: Visualize the original audio and the trimmed audio


In [ ]:
#<TODO>: Complete the function to add random noise to the audio
def add_random_noise(audio, noise_factor=0.05):

    # Add Noise
    noise = np.random.randn(len(audio))
    noisy_audio = None # Todo

    return noisy_audio

#<TODO>: Load and Add Noise to the first audio

#<TODO>: Play the noisy audio

#<TODO>: Visualize the original audio and the noisy audio


In [ ]:
#<TODO>: Complete this function to generate spectrogram
def audio_to_mel_spectrogram(audio, sample_rate=22050):


    # Generate spectrogram
    mel_spec = None # Todo


    # Convert amplitudes to decibels
    mel_spec_db = None # Todo

    return mel_spec_db

#<TODO>: Generate and visualize the spectrogram for the first audio


#<TODO>: Generate other mel spectrograms for other audio classes such as guns,children playing, and compare them

In [ ]:
import numpy as np

# <TODO>: Complete this function to prepare all spectrogram data for CNN training
def prepare_cnn_data(metadata, sample_rate=22050, max_pad_len=173):
    X = []
    y = []

    for i in range(len(metadata)):
        file_path = metadata.iloc[i]["audio_path"]
        label = metadata.iloc[i]["classID"]

        # Load audio
        audio, _ = None # Todo

        # Trim audio files
        trimmed_audio, _ = None # Todo

        # Add noise to audio files
        noisy_audio = None # Todo

        # Convert audio file to Mel spectrogram
        mel_spec = None # Todo

        # Pad or truncate spectrogram to a fixed length
        if mel_spec.shape[1] < max_pad_len:
            # Pad with zeros
            pad_width = max_pad_len - mel_spec.shape[1]
            mel_spec = np.pad(mel_spec, ((0, 0), (0, pad_width)), mode='constant')
        elif mel_spec.shape[1] > max_pad_len:
            # Truncate
            mel_spec = mel_spec[:, :max_pad_len]

        # Add spectrogram to X
        X.append(mel_spec)

        # Add label to y
        y.append(label)

    # Convert lists to NumPy arrays
    X = np.array(X)
    y = np.array(y)

    # Add channel dimension for CNN
    # Shape changes from: (samples, height, width)
    # To: (samples, height, width, channels)
    X = X[..., np.newaxis]

    return X, y

In [ ]:
# Load the dataset
full_metadata = metadata.reset_index(drop=True)

X, y = prepare_cnn_data(full_metadata)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (300, 128, 173, 1)
y shape: (300,)


In [ ]:
""" Splitting data into training and testing"""

# converting the labels into one-hot encoding
num_classes = metadata["classID"].nunique()
y_categorical = to_categorical(y, num_classes=num_classes)

# Splitting training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_categorical,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
""" CNN model architecture """

#<TODO>: Complete a CNN architecture
model = None

model.summary()

In [ ]:
#<TODO>: Complete the training loop
model.compile(
    optimizer=None,
    loss=None,
    metrics=None
)

history = model.fit(
    X_train,
    y_train,
    epochs=None,
    batch_size=None,
    validation_split=None
)

In [ ]:
#<TODO>: Complete the test evaluation
test_loss, test_accuracy = None

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)